In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [0]:
spark = SparkSession.builder.appName("cleanLoanRepaymentData").getOrCreate()

In [0]:
repayment_schema = 'loan_id string, total_principal_received decimal(10, 2),total_interest_received decimal(10, 2),total_late_fee_received decimal(10, 2),total_payment_received decimal(10, 2),last_payment_amount decimal(10, 2),last_payment_date string,next_payment_date string'

In [0]:
repayment_df = spark.read.csv("/Volumes/lending_cart_dev/bronze/bronze_volume/raw/loans_repayments_csv/", header = True, schema = repayment_schema)

In [0]:
display(repayment_df.filter("total_principal_received is null"))

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date
Total amount funded in policy code 1: 2050909275,null,null,null,null,null,null,null
Total amount funded in policy code 2: 820109297,null,null,null,null,null,null,null
Total amount funded in policy code 1: 2080429200,null,null,null,null,null,null,null
Total amount funded in policy code 2: 737901574,null,null,null,null,null,null,null
839247,null,10.0,0.0,0.0,null,27.0,0.0
704365,null,0.0,0.0,52.4,251.77,18000.0,0.0
537783,null,null,8.0,699.0,16.0,92.9,f
529331,null,0.0,0.0,57.5,7259.87,13625.0,0.0
527801,null,0.0,0.0,10.4,1022.49,9000.0,0.0
521687,null,10.0,0.0,6.0,null,35.0,0.0


In [0]:
columns_subset = ["total_principal_received","total_interest_received","total_late_fee_received","total_payment_received","last_payment_amount"]

In [0]:
repayment_remove_nulls_df = repayment_df.na.drop(subset = columns_subset)

In [0]:
display(repayment_remove_nulls_df.filter("total_payment_received = 0 and total_principal_received !=0")) 

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date
1064185,11600.98,11600.98,10000.00,0.00,0.00,0.0,Dec-2014
516382,21890.23,21856.03,16000.00,0.00,0.00,0.0,Mar-2014
528899,3045.04,3019.64,2500.00,0.00,0.00,0.0,Jan-2013
527598,2398.91,2220.51,2200.00,0.00,0.00,0.0,Jul-2011
525697,21797.86,19894.90,15750.00,0.00,0.00,0.0,Jun-2015
522641,3146.82,3146.82,3000.00,0.00,0.00,0.0,Sep-2011
515655,29938.58,29905.75,22800.00,0.00,0.00,0.0,May-2013
501234,15219.31,15155.90,12000.00,0.00,0.00,0.0,May-2013
498194,11642.71,11031.47,10000.00,0.00,0.00,0.0,Jan-2013
495171,11138.84,10024.96,10000.00,0.00,0.00,0.0,Apr-2013


In [0]:
display(repayment_remove_nulls_df.filter("loan_id = '141581221'"))

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date
141581221,1055.81,2591.70,0.00,3647.51,709.23,Mar-2019,Apr-2019


In [0]:
repayment_payreceived_df = repayment_remove_nulls_df.withColumn("total_payment_received2",
                                     when(col("total_payment_received") != 0, col("total_payment_received") )
                                     .otherwise(col("total_principal_received") + col("total_interest_received") + col("total_late_fee_received"))
                                    )



In [0]:
display(repayment_payreceived_df.filter("total_payment_received = 0"))

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,total_payment_received2
141313188,0.00,0.00,0.00,0.00,0.00,null,null,0.00
141013155,0.00,0.00,0.00,0.00,0.00,null,null,0.00
141415504,0.00,0.00,0.00,0.00,0.00,null,null,0.00
141242794,0.00,0.00,0.00,0.00,0.00,null,null,0.00
141422742,0.00,0.00,0.00,0.00,0.00,null,null,0.00
141305199,0.00,0.00,0.00,0.00,0.00,null,null,0.00
140438338,0.00,0.00,0.00,0.00,0.00,null,null,0.00
140621549,0.00,0.00,0.00,0.00,0.00,null,null,0.00
141320790,0.00,0.00,0.00,0.00,0.00,null,null,0.00
141241575,0.00,0.00,0.00,0.00,0.00,null,null,0.00


In [0]:
repayment_notzero_df = repayment_payreceived_df.filter("not(total_principal_received=0 or total_interest_received=0 or total_payment_received2=0)")

In [0]:
#display(repayment_notzero_df)
display(repayment_notzero_df.filter("last_payment_amount =0"))
#display(repayment_payreceived_df.filter("total_payment_received = 0"))

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,total_payment_received2
134610293,25000.00,1463.63,0.00,26463.63,0.00,Jan-2019,null,26463.63
134434053,6000.00,353.97,0.00,6353.97,0.00,Feb-2019,null,6353.97
132926182,20000.00,519.98,0.00,20519.98,0.00,Aug-2018,null,20519.98
132233232,7000.00,551.23,0.00,7551.23,0.00,Sep-2018,null,7551.23
131344770,12000.00,271.02,0.00,12271.02,0.00,Jul-2018,null,12271.02
1064185,11600.98,11600.98,10000.00,0.00,0.00,0.0,Dec-2014,33201.96
516382,21890.23,21856.03,16000.00,0.00,0.00,0.0,Mar-2014,59746.26
529353,1.00,7288.00,47.00,109.00,0.00,0.0,21221.5644086071,109.00
528899,3045.04,3019.64,2500.00,0.00,0.00,0.0,Jan-2013,8564.68
527598,2398.91,2220.51,2200.00,0.00,0.00,0.0,Jul-2011,6819.42


In [0]:
display(repayment_notzero_df.select("last_payment_date","next_payment_date") \
                            .filter(~col("last_payment_date").rlike("[A-Z][a-z]{2}-\\d{4}$"))
        )

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5458903464175702>, line 1
----> 1 display(repayment_notzero_df.select("last_payment_date","last_payment_date_trans") \
      2                             .filter(~col("last_payment_date").rlike("[A-Z][a-z]{2}-\\d{4}$"))
      3         )

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was mis

In [0]:

repayment_fixeddates_df = ( repayment_notzero_df
                                .withColumn("last_payment_date_trans",
                                # If value is like Jan-2026 or Feb-2025 then keep it otherwise replace with None
                                when(col("last_payment_date").rlike("[A-Z][a-z]{2}-\\d{4}$"),
                                      col("last_payment_date")
                                )
                                .otherwise(None)
                                ) 
                                .withColumn("next_payment_date_trans",
                                # If value is like Jan-2026 or Feb-2025 then keep it otherwise replace with None
                                when(col("next_payment_date").rlike("[A-Z][a-z]{2}-\\d{4}$"),
                                      col("next_payment_date")
                                )
                                .otherwise(None)
                                )
                            )

In [0]:
display(repayment_fixeddates_df.select("last_payment_date","last_payment_date_trans") \
                            .filter(~col("last_payment_date").rlike("[A-Z][a-z]{2}-\\d{4}$"))
        )

last_payment_date,last_payment_date_trans
0.0,null
0.0,null
0.0,null
0.0,null
0.0,null
0.0,null
0.0,null
0.0,null
0.0,null
0.0,null


In [0]:
display(repayment_fixeddates_df.select("next_payment_date","next_payment_date_trans") \
                            .filter(~col("last_payment_date").rlike("[A-Z][a-z]{2}-\\d{4}$"))
        )

next_payment_date,next_payment_date_trans
Dec-2014,Dec-2014
Mar-2014,Mar-2014
21221.5644086071,null
Jan-2013,Jan-2013
Jul-2011,Jul-2011
Jun-2015,Jun-2015
Sep-2011,Sep-2011
May-2013,May-2013
14135.97,null
May-2013,May-2013


In [0]:
repayment_fixeddates_df.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- total_principal_received: decimal(10,2) (nullable = true)
 |-- total_interest_received: decimal(10,2) (nullable = true)
 |-- total_late_fee_received: decimal(10,2) (nullable = true)
 |-- total_payment_received: decimal(10,2) (nullable = true)
 |-- last_payment_amount: decimal(10,2) (nullable = true)
 |-- last_payment_date: string (nullable = true)
 |-- next_payment_date: string (nullable = true)
 |-- total_payment_received2: decimal(12,2) (nullable = true)
 |-- last_payment_date_trans: string (nullable = true)
 |-- next_payment_date_trans: string (nullable = true)



In [0]:
repayment_final_df = repayment_fixeddates_df.drop("total_payment_received","last_payment_date","next_payment_date") \
                        .withColumnRenamed("total_payment_received2","total_payment_received") \
                        .withColumnRenamed("last_payment_date_trans","last_payment_date") \
                        .withColumnRenamed("next_payment_date_trans","next_payment_date")

In [0]:
display(repayment_final_df)

loan_id,total_principal_received,total_interest_received,total_late_fee_received,last_payment_amount,total_payment_received,last_payment_date,next_payment_date
141581221,1055.81,2591.70,0.00,709.23,3647.51,Mar-2019,Apr-2019
141506948,1252.75,306.04,0.00,312.63,1558.79,Mar-2019,Apr-2019
141357400,626.37,354.96,0.00,197.27,981.33,Mar-2019,Apr-2019
139445427,1118.16,297.36,0.00,283.95,1415.52,Mar-2019,Apr-2019
141407409,1169.72,3605.30,0.00,964.90,4775.02,Mar-2019,Apr-2019
141360802,2313.98,2512.88,0.00,952.02,4826.86,Mar-2019,Apr-2019
141163960,4689.63,1994.93,0.00,1342.57,6684.56,Mar-2019,Apr-2019
141533932,585.29,640.53,15.00,235.13,1240.82,Mar-2019,Apr-2019
141441276,2030.82,762.81,0.00,477.62,2793.63,Mar-2019,Apr-2019
141569080,1803.55,1110.59,0.00,585.91,2914.14,Mar-2019,Apr-2019


In [0]:
repayment_final_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .option("path","/Volumes/lending_cart_dev/bronze/bronze_volume/cleaned/loans_repayments/") \
    .save()